In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')


if openai_api_key:
    print(f"OpenAI API Key exist and begines with {openai_api_key[:6]}")
else:
    print("OpenAI API key not set")

In [3]:
openai = OpenAI()
ollama_url="http://localhost:11434/v1"
ollama= OpenAI(api_key="ollama", base_url=ollama_url)

In [4]:
def ollama_agent(model, messages):
    response = ollama.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

In [5]:
def openai_agent(messages):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages = messages
    )
    return response.choices[0].message.content

In [6]:
def self_critique(agent, draft):

    critique_prompt = [
        {
            "role": "system",
            "content": f"""You are reviewing your own response as {agent['name']}.
            Critically evaluate the proposal.

            Identify:
            - Weak assumptions
            - Operational risks
            - Missing considerations
            - Better alternatives
            """
        },
        {
            "role": "user",
            "content": draft
        }
    ]

    critique = ollama_agent(agent["model"], critique_prompt)

    revision_prompt = [
        {
            "role": "system",
            "content": f"""Revise the original response using the critique.

            Return a stronger structured manufacturing recommendation
            using the required format:

            OBSERVATIONS
            PROPOSED ACTIONS
            MACHINE ALLOCATION
            RISKS
            EXPECTED IMPACT
            """
        },
        {
            "role": "user",
            "content": f"Original:\n{draft}\n\nCritique:\n{critique}"
        }
    ]

    improved = ollama_agent(agent["model"], revision_prompt)

    return improved

In [ ]:
# Define team members with refined roles and prompts
team_members = [
    {
        "name": "Boss",
        "type": "ollama",
        "model":"llama3.2",
        "system": """You are the boss: decisive, results-oriented, and no-nonsense. 
        Your primary concern is ROI, throughput, and operational efficiency. 
        Keep recommendations concise and focused on high-impact decisions."""
    },
    {
        "name": "Engineer",
        "type": "openai",
        "model":"llama3.2",
        "system": """You are a manufacturing engineer and problem solver. 
        Your goal is to improve efficiency, reduce cost, and optimize laser throughput. 
        Explain technical ideas clearly, referencing laser capabilities:
        - 12kW laser: fastest on thin gauges on thicker gauge there is little benefits. Stainless Steel must be handled carefully and not mark material.
        - 10kW and 8kW lasers: comparable speed on thicker gauges; 8kW has an LiftMaster, 10kW LiftMasterCompact upgrade planned within 3 months.
        - 8kW the cycle time for the LiftMaster is 4 minutes.
        - 10kW the cycle time of the LiftMasterCompace is 3-1/2 minutes.
        - 12kW the cycle time is 20-25 minutes per sheet because it is denested at the machine.
        - 8kW uses different nozzles.  
        - 10kW and 12kW uses same nozzles.
        Provide actionable suggestions for layout, nesting, cutting strategy, tailored to each gauge and laser power.
        The LiftMasters will prevent damaging Stainless Steel which is preferred."""
    },
    {
        "name": "Reviewer",
        "type": "openai",
        "model":"llama3.2",
        "system": """You are a fellow Manufacturing Engineer reviewer and process analyst. 
        Your role is to evaluate ideas objectively, suggest improvements, 
        and point out potential risks or inefficiencies. 
        Always provide constructive feedback and practical alternatives."""
    }
]

In [ ]:
# Initialize conversation
conversation = []
user_prompt = """We are facing a line balancing problem on our three Trumpf TruLaser 5040 fiber lasers: 12kW, 10kW, and 8kW. 

We manufacture parts from multiple materials, in order of volume: 
1. Mild steel  
2. High Tensile (HT) steel  
3. Stainless steel  
4. Abrasion Resistant (AR) steel  

The parts vary in thickness, from most common to least common: 
- 0.25 in, 0.38 in, 0.5 in, 0.63 in, 0.75 in, 1 in  
- 16 gauge, 14 gauge, 12 gauge, 10 gauge  

Please provide recommendations on line balancing, throughput optimization, and how to allocate parts efficiently across the three lasers."""

conversation.append({"role": "user", "content": user_prompt})

rounds = 2

for r in range(rounds):

    display(Markdown(f"## ROUND {r+1}"))

    for member in team_members:

        messages = [
            {"role": "system", "content": member["system"]},
            *conversation[-10:]
        ]

        draft = ollama_agent(member["model"], messages)

        improved = self_critique(member, draft)

        display(Markdown(f"**{member['name']}**"))
        display(Markdown(improved))

        conversation.append({
            "role": "assistant",
            "content": f"{member['name']}:\n{improved}"
        })
        

In [ ]:
final_prompt = [
    {
        "role": "system",
        "content": """You are a senior manufacturing strategist.

        Review the team discussion and produce the final operational
        plan for balancing the three Trumpf lasers.

        Provide:

        FINAL MACHINE ASSIGNMENT
        MATERIAL FLOW STRATEGY
        BOTTLENECKS
        EXPECTED THROUGHPUT IMPROVEMENT
        IMPLEMENTATION STEPS
        """
    },
    *conversation
]

final_answer = openai_agent(final_prompt)

display(Markdown("## FINAL DECISION"))
display(Markdown(final_answer))